[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CkCreative/transformers-and-math-notebooks/blob/main/11-training-across-many-gpus.ipynb)

# Module 11: Training Across Many GPUs

Companion notebook to [Module 11](https://transformers-and-math.drmhse.com/modules/11-training-across-many-gpus) of *Transformers & Math*. Run the cells in order; each part names the section of the module it goes with. Everything runs on a free Colab CPU in under a minute or two.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.checkpoint import checkpoint

## Part 1 — Where the memory goes (sections 11.1 and 11.3)

In [ ]:
def state_gb(params, gpus=1, zero_stage=0):
    weights, grads, master, adam = 2, 2, 4, 8          # bytes per parameter
    if zero_stage >= 1: master, adam = master / gpus, adam / gpus
    if zero_stage >= 2: grads = grads / gpus
    if zero_stage >= 3: weights = weights / gpus
    return params * (weights + grads + master + adam) / 1e9

for stage in range(4):
    print(f'70B on 64 GPUs, ZeRO stage {stage}: {state_gb(70e9, 64, stage):7.1f} GB per GPU')

## Part 2 — Tensor parallelism is exact (section 11.4)
Split $W_1$ by columns and $W_2$ by rows across $t$ "GPUs"; the partial results sum to the full answer.

In [ ]:
torch.manual_seed(0)
d, dff, t = 64, 256, 4
x, W1, W2 = torch.randn(10, d), torch.randn(d, dff), torch.randn(dff, d)
full = torch.relu(x @ W1) @ W2
parts = [torch.relu(x @ W1[:, i::t]) @ W2[i::t, :] for i in range(t)]
print('sum of shards equals the full layer:', torch.allclose(sum(parts), full, atol=1e-3))

## Part 3 — The pipeline bubble (section 11.5)

In [ ]:
for p, m in [(4, 1), (4, 8), (4, 28), (8, 32)]:
    print(f'{p} stages, {m:2} micro-batches: bubble = {(p - 1) / (m + p - 1):.1%}')

## Part 4 — Activation checkpointing, measured (section 11.6)
Same model, same result; less memory kept for the backward pass, more compute.

In [ ]:
class Deep(nn.Module):
    def __init__(self, L=24, d=512, ckpt=False):
        super().__init__()
        self.layers = nn.ModuleList([nn.Sequential(nn.Linear(d, d), nn.GELU()) for _ in range(L)])
        self.ckpt = ckpt
        self.calls = 0                       # layer forward passes, including recomputation
    def run(self, f, x):
        self.calls += 1
        return f(x)
    def forward(self, x):
        for f in self.layers:
            x = checkpoint(self.run, f, x, use_reentrant=False) if self.ckpt else self.run(f, x)
        return x

def saved_bytes(model, x):
    total = 0
    def pack(t):
        nonlocal total
        total += t.numel() * t.element_size()
        return t
    with torch.autograd.graph.saved_tensors_hooks(pack, lambda t: t):
        out = model(x).sum()
    return total, out

x = torch.randn(256, 512)
for ckpt in (False, True):
    torch.manual_seed(0)
    model = Deep(ckpt=ckpt)
    mem, out = saved_bytes(model, x)
    out.backward()
    print(f'checkpointing={ckpt!s:5}  tensors saved for backward: {mem / 1e6:5.1f} MB   layer forward passes: {model.calls}')

## Your turn
1. With ZeRO stage 1 on 8 GPUs, what is the per-GPU state of a 7B model? (Answer: 38.5 GB.)
2. How many micro-batches does an 8-stage pipeline need to keep the bubble under 10%?